In [1]:
import os
os.environ["HF_HOME"] = "/home/jovyan/.cache/huggingface"

import torch
import polars as pl
from pathlib import Path
import time
import numpy as np
from evo2 import Evo2
import psutil # troubleshooting

# configuration
DATA_ROOT = Path.home() / "vambersky_t/Data"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"


LAYER = "blocks.28.mlp.l3"
SEQ_LEN = 10_000
BIN_SIZE = 50
N_BINS = SEQ_LEN // BIN_SIZE
EMB_DIM = 4096

TFS_TO_PROCESS = Path("tfs_to_process.txt").read_text().splitlines()
CHECKPOINT_EVERY = 500

assert DATA_ROOT.exists(),   f"Data root missing: {DATA_ROOT}"
assert WINDOWS_DIR.exists(), f"Windows dir missing: {WINDOWS_DIR}"
assert N_BINS == 200

print(f"Bins: {N_BINS} × {BIN_SIZE} bp = {SEQ_LEN} bp")
print(f"Per-peak tensor: ({N_BINS}, {EMB_DIM}) float16")
print(f"Checkpoint interval: every {CHECKPOINT_EVERY} peaks")

# load model
print(f"Loading model...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = Evo2("evo2_7b")
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Bins: 200 × 50 bp = 10000 bp
Per-peak tensor: (200, 4096) float16
Checkpoint interval: every 500 peaks
Loading model...


[04/23/26 12:22:08] INFO     httpx - INFO - HTTP Request: GET                                       ]8;id=14818726;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14818727;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/arcinstitute/evo2_7b/revision/main                  
                             "HTTP/1.1 200 OK"                                                                     

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/home/jovyan/envs/evo2/lib/python3.12/site-packages/evo2/models.py:282: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(


...

                    INFO     StripedHyena - INFO - Initialized model                                   ]8;id=14819141;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=14819142;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#691\691]8;;\

                    INFO     vortex.model.utils - INFO - Loading                                        ]8;id=14819149;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/utils.py\utils.py]8;;\:]8;id=14819150;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/utils.py#92\92]8;;\
                             /home/jovyan/.cache/huggingface/hub/models--arcinstitute--evo2_7b/snapshot            
                             s/bda0089f92582d5baabf0f22d9fc85f3588f6b58/evo2_7b.pt                                 

Extra keys in state_dict: {'blocks.10.mixer.dense._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.13.mixer.mixer.filter.t', 'blocks.20.projections._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.30.mixer.mixer.filter.t', 'blocks.10.mixer.attn._extra_state', 'blocks.18.projections._extra_state', 'blocks.13.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.23.projections._extra_state', 'blocks.27.projections._extra_state', 'blocks.11.projections._extra_state', 'blocks.8.projections._extra_state', 'unembed.weight', 'blocks.2.mixer.mixer.filter.t', 'blocks.15.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.9.projections._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.26.projections._extra_state', 'blocks.27.mixer.mixer.filter.t', 'blocks.16.projections._extra_state', 'blocks.22.projections._extra_state', 'blocks.1.projections._extra_state', 'blo

[04/23/26 12:22:10] INFO     StripedHyena - INFO - Adjusting Wqkv for column split (permuting rows)    ]8;id=14819156;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=14819157;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#976\976]8;;\

Model loaded in 5.4s
VRAM allocated: 13.17 GB


In [2]:
# get the embeddings - mean of every 50, non overlapping (probrat s Vojtou - rozlišení, porovnatelnost)

def embed_sequence(sequence: str) -> np.ndarray:
    """
    Forward-pass one 10 kb sequence through Evo 2, extract embeddings
    from LAYER, bin into (200, 4096) via reshape + mean, return as
    float16 numpy array.

    Uses module-level `model` and `LAYER`.
    """
    input_ids = torch.tensor(
        model.tokenizer.tokenize(sequence),
        dtype=torch.int,
    ).unsqueeze(0).to("cuda:0")

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[LAYER],
        )

    emb = embeddings[LAYER][0]                 # (10000, 4096)

    # reshape + mean - as i did in the test, this works.
    binned = emb.reshape(N_BINS, BIN_SIZE, -1).mean(dim=1)

    return binned.cpu().to(torch.float16).numpy()

# test, delete
test_df = pl.read_csv(
    next((WINDOWS_DIR / "CTCF").glob("*.full_peaks.csv.gz")),
    n_rows=1,
)
test_emb = embed_sequence(test_df["sequence"][0])
print(test_emb.shape, test_emb.dtype) 

(200, 4096) float16


In [3]:
# don't mess up my storage, pls

def get_output_paths(csv_path: Path) -> tuple[Path, Path, Path]:
    """
    Derive (final_npz, final_parquet, checkpoint_npz) from input CSV path.
    """
    base_name = csv_path.name.split(".full_peaks")[0]
    tf = base_name.split("__")[1]

    out_dir = EMBEDDINGS_DIR / tf
    out_dir.mkdir(parents=True, exist_ok=True)

    final_npz      = out_dir / f"{base_name}.embeddings.npz"
    final_parquet   = out_dir / f"{base_name}.peak_ids.parquet"
    checkpoint_npz  = out_dir / f"{base_name}.embeddings.checkpoint.npz"

    return final_npz, final_parquet, checkpoint_npz


In [4]:
def process_csv_file(csv_path: Path) -> None:
    """
    Full embedding extraction for one CSV file of peak sequences.
    """
    final_npz, final_parquet, checkpoint_npz = get_output_paths(csv_path)

    if final_npz.exists() and final_parquet.exists(): # skip processed
        print(f"  SKIP (complete): {final_npz.name}")
        return
    
    df = pl.read_csv(csv_path)
    n_peaks = len(df)
    sequences = df["sequence"].to_list()
    peak_ids  = df["peak_id"].to_list()

    bad = [i for i, s in enumerate(sequences) if len(s) != SEQ_LEN] # check if length = 10 000 bp
    if bad:
        print(f"  WARNING: {len(bad)} sequences != {SEQ_LEN} bp "
              f"(indices {bad[:5]}{'...' if len(bad) > 5 else ''}). "
              f"These will be skipped.")

    work_path = final_npz.parent / f"{final_npz.stem}.work.npy"
    progress_path = final_npz.parent / f"{final_npz.stem}.progress.txt"

    if work_path.exists() and progress_path.exists():
        progress = progress_path.read_text().strip().split("\n")
        valid_peak_ids = progress[:-1]       # all lines except last
        start_idx      = int(progress[-1])   # last line is next_idx
        n_written      = len(valid_peak_ids)
        print(f"  RESUME: {start_idx}/{n_peaks} peaks scanned, "
              f"{n_written} valid embeddings on disk")
        mmap = np.memmap(work_path, dtype=np.float16, mode="r+",
                         shape=(n_peaks, N_BINS, EMB_DIM))
    else:
        valid_peak_ids = []
        start_idx      = 0
        n_written      = 0
        mmap = np.memmap(work_path, dtype=np.float16, mode="w+",
                         shape=(n_peaks, N_BINS, EMB_DIM))


    # the new LOOP!
    t0 = time.time()
    for i in range(start_idx, n_peaks):
        seq = sequences[i]

        if len(seq) != SEQ_LEN:
            continue

        emb = embed_sequence(seq)
        mmap[n_written] = emb        # Claude thinks this will help with RAM
        n_written += 1
        valid_peak_ids.append(peak_ids[i])

        if (i + 1) % 100 == 0:
            elapsed   = time.time() - t0
            done      = i + 1 - start_idx
            rate      = done / elapsed
            remaining = (n_peaks - i - 1) / rate if rate > 0 else float("inf")
            ram_gb    = psutil.Process().memory_info().rss / 1e9
            print(f"  [{i+1}/{n_peaks}] {rate:.1f} peaks/s  "
                  f"~{remaining / 60:.0f} min left  "
                  f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB  "
                  f"RAM {ram_gb:.1f} GB")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            ckpt_t0 = time.time()
            mmap.flush()
            # Progress file: peak IDs + next index to scan
            progress_path.write_text(
                "\n".join(valid_peak_ids) + "\n" + str(i + 1)
            )
            print(f"  CHECKPOINT at peak {i+1} ({time.time() - ckpt_t0:.1f}s)")
        

    # finalize
    mmap.flush()
    del mmap

    # Trim to actual number of valid peaks and compress
    save_t0 = time.time()
    full = np.memmap(work_path, dtype=np.float16, mode="r",
                     shape=(n_peaks, N_BINS, EMB_DIM))
    trimmed = np.array(full[:n_written])    # only valid rows, in RAM briefly
    del full
    np.savez_compressed(final_npz, embeddings=trimmed)
    print(f"  Final save took {time.time() - save_t0:.1f}s")
    del trimmed

    sidecar = (
        pl.DataFrame({"peak_id": valid_peak_ids})
        .join(
            df.select(["peak_id", "chr", "start", "end", "center"]),
            on="peak_id",
            how="left",
        )
    )
    sidecar.write_parquet(final_parquet)

    elapsed_total = time.time() - t0
    print(f"  DONE: {n_written} peaks → {final_npz.name}  "
          f"{elapsed_total / 60:.1f} min")

    # Clean up working files
    work_path.unlink()
    progress_path.unlink(missing_ok=True)
    print(f"  Working files removed")

In [5]:
csv_files = []
for entry in TFS_TO_PROCESS:
    tf = entry.split("__")[1]
    path = WINDOWS_DIR / tf / f"{entry}.full_peaks.csv.gz"
    if not path.exists():
        print(f"WARNING: Not found: {path.name}")
        continue
    csv_files.append(path)

print(f"\nTotal: {len(csv_files)} files")
for f in csv_files:
    print(f"  {f.parent.name}/{f.name}")


Total: 9 files
  MYC/ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz
  MYC/ENCFF700CXD__MYC__H1.full_peaks.csv.gz
  MYC/ENCFF149BIS__MYC__MCF-7.full_peaks.csv.gz
  CTCF/ENCFF692RPA__CTCF__H1.full_peaks.csv.gz
  CTCF/ENCFF769AUF__CTCF__K562.full_peaks.csv.gz
  CTCF/ENCFF017XLW__CTCF__GM12878.full_peaks.csv.gz
  SPI1/ENCFF881ARS__SPI1__GM12878.full_peaks.csv.gz
  SPI1/ENCFF198SPL__SPI1__GM12878.full_peaks.csv.gz
  SPI1/ENCFF185DGL__SPI1__K562.full_peaks.csv.gz


In [6]:
# npz, parquet, _ = get_output_paths(test_csv)
# data = np.load(npz)
# ids = pl.read_parquet(parquet)
# print(data["embeddings"].shape, data["embeddings"].dtype)
# print(ids.shape)
# print(ids.head())


In [7]:
# files_to_run = [
#     WINDOWS_DIR / "MYC"  / "ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz",
#     WINDOWS_DIR / "MYC"  / "ENCFF700CXD__MYC__H1.full_peaks.csv.gz",
#     WINDOWS_DIR / "CTCF" / "ENCFF017XLW__CTCF__GM12878.full_peaks.csv.gz",
#     WINDOWS_DIR / "RANDOM" / "RANDOM__RANDOM__hg19.full_peaks.csv.gz",
# ]

# for i, csv_path in enumerate(files_to_run):
#     print(f"\n[{i+1}/{len(files_to_run)}] {csv_path.parent.name}/{csv_path.name}")
#     process_csv_file(csv_path)
#     torch.cuda.empty_cache()

In [ ]:
pipeline_t0 = time.time()
for idx, csv_path in enumerate(csv_files):
    print(f"\n[{idx+1}/{len(csv_files)}] {csv_path.parent.name}/{csv_path.name}")
    process_csv_file(csv_path)
    torch.cuda.empty_cache()

elapsed = time.time() - pipeline_t0
print(f"\n{'='*50}")
print(f"Pipeline complete: {elapsed / 3600:.1f} hours")



[1/9] MYC/ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz
  SKIP (complete): ENCFF765CKK__MYC__GM12878.embeddings.npz

[2/9] MYC/ENCFF700CXD__MYC__H1.full_peaks.csv.gz
  SKIP (complete): ENCFF700CXD__MYC__H1.embeddings.npz

[3/9] MYC/ENCFF149BIS__MYC__MCF-7.full_peaks.csv.gz
  SKIP (complete): ENCFF149BIS__MYC__MCF-7.embeddings.npz

[4/9] CTCF/ENCFF692RPA__CTCF__H1.full_peaks.csv.gz
  RESUME: 9500/9961 peaks scanned, 9500 valid embeddings on disk
  [9600/9961] 0.2 peaks/s  ~35 min left  VRAM 13.2 GB  RAM 1.8 GB
  [9700/9961] 0.2 peaks/s  ~25 min left  VRAM 13.2 GB  RAM 2.0 GB
  [9800/9961] 0.2 peaks/s  ~14 min left  VRAM 13.2 GB  RAM 2.1 GB
  [9900/9961] 0.2 peaks/s  ~5 min left  VRAM 13.2 GB  RAM 2.3 GB
